# 04 - Text Extraction (OCR)

This notebook shows how to use OCI Vision's text detection (OCR) feature.
Since demo data for OCR is not cached yet, we walk through the API usage
pattern and explain the response structure so you can apply it with real
OCI credentials.

## Setup

In [ ]:
# !pip install oci-vision-ai[notebooks]

from oci_vision.core.client import VisionClient

# Demo mode -- OCR data is not cached, so we show the pattern
client = VisionClient(demo=True)
print(f"Client ready (demo={client.is_demo})")

## Run the analysis

The `detect_text()` method sends an image to the OCI Vision TEXT_DETECTION
feature. In demo mode this returns `None` because no OCR demo data is cached
yet. Below we show the call pattern and then manually construct an example
response to demonstrate the data model.

In [ ]:
# This is how you call OCR -- returns None in demo mode (no cached data)
result = client.detect_text("dog_closeup.jpg")
print(f"detect_text() returned: {result}")
print()
print("In live mode with real credentials, this would return a")
print("TextDetectionResult containing all detected text lines and words.")

## Explore the results

Below we construct a synthetic `TextDetectionResult` to demonstrate
the data model and how you would work with real OCR results.

In [ ]:
from oci_vision.core.models import (
    TextDetectionResult, TextLine, TextWord,
    BoundingPolygon, Vertex
)

# Synthetic example -- what a real response looks like
example_result = TextDetectionResult(
    model_version="1.5.97",
    lines=[
        TextLine(
            text="STOP",
            confidence=0.9912,
            bounding_polygon=BoundingPolygon(
                normalized_vertices=[
                    Vertex(x=0.35, y=0.20),
                    Vertex(x=0.65, y=0.20),
                    Vertex(x=0.65, y=0.35),
                    Vertex(x=0.35, y=0.35),
                ]
            ),
            words=[
                TextWord(
                    text="STOP",
                    confidence=0.9912,
                    bounding_polygon=BoundingPolygon(
                        normalized_vertices=[
                            Vertex(x=0.35, y=0.20),
                            Vertex(x=0.65, y=0.20),
                            Vertex(x=0.65, y=0.35),
                            Vertex(x=0.35, y=0.35),
                        ]
                    )
                )
            ]
        ),
        TextLine(
            text="Speed Limit 30",
            confidence=0.9567,
            bounding_polygon=BoundingPolygon(
                normalized_vertices=[
                    Vertex(x=0.10, y=0.70),
                    Vertex(x=0.45, y=0.70),
                    Vertex(x=0.45, y=0.80),
                    Vertex(x=0.10, y=0.80),
                ]
            ),
            words=[
                TextWord(text="Speed", confidence=0.9734,
                         bounding_polygon=BoundingPolygon(
                             normalized_vertices=[Vertex(x=0.10,y=0.70), Vertex(x=0.22,y=0.70),
                                                  Vertex(x=0.22,y=0.80), Vertex(x=0.10,y=0.80)])),
                TextWord(text="Limit", confidence=0.9612,
                         bounding_polygon=BoundingPolygon(
                             normalized_vertices=[Vertex(x=0.23,y=0.70), Vertex(x=0.35,y=0.70),
                                                  Vertex(x=0.35,y=0.80), Vertex(x=0.23,y=0.80)])),
                TextWord(text="30", confidence=0.9356,
                         bounding_polygon=BoundingPolygon(
                             normalized_vertices=[Vertex(x=0.36,y=0.70), Vertex(x=0.45,y=0.70),
                                                  Vertex(x=0.45,y=0.80), Vertex(x=0.36,y=0.80)])),
            ]
        ),
    ]
)

print(f"Lines detected: {len(example_result.lines)}")
print(f"Full text:\n{example_result.full_text}")

In [ ]:
# Iterate over lines and words
for i, line in enumerate(example_result.lines, 1):
    print(f"Line {i}: \"{line.text}\" (confidence: {line.confidence:.1%})")
    print(f"  Position: {line.bounding_polygon.human_position(800, 600)}")
    for w, word in enumerate(line.words, 1):
        print(f"  Word {w}: \"{word.text}\" ({word.confidence:.1%})")
    print()

## Visualize

The `render_overlay` function draws yellow highlights on detected text
regions. Here we show what that would look like with our synthetic data.

In [ ]:
from oci_vision.core.models import AnalysisReport
from oci_vision.core.renderer import render_overlay
from PIL import Image

# Create a placeholder image and render the OCR overlay
placeholder = Image.new('RGB', (800, 600), color=(240, 240, 240))

report = AnalysisReport(
    image_path="synthetic_example.jpg",
    text=example_result
)

overlay = render_overlay(placeholder, report)
print(f"Overlay rendered: {overlay.size[0]}x{overlay.size[1]}")
print("Yellow highlights mark detected text regions.")
overlay

## Under the hood

In [ ]:
import json

# What the raw response looks like
raw = example_result.model_dump()
print(json.dumps(raw, indent=2, default=str))

### Live API usage pattern

When using real OCI credentials, the call is identical:

```python
client = VisionClient()  # uses ~/.oci/config
result = client.detect_text("path/to/image.jpg")

# Or from Object Storage
result = client.detect_text("oci://my-bucket/image.jpg")

print(result.full_text)  # all text concatenated
for line in result.lines:
    print(f"{line.text} ({line.confidence:.1%})")
```

## Try it yourself

1. **Live OCR**: Switch to `VisionClient()` and pass an image containing
   text (receipts, signs, documents) to `detect_text()`.
2. **Word-level analysis**: Iterate over `line.words` to get per-word
   bounding boxes and confidence scores.
3. **Full pipeline**: Combine OCR with classification to both identify
   what is in an image and read any text it contains.
4. **Bounding boxes**: Use `to_pixels(w, h)` on word bounding polygons
   to crop individual words from the source image.